In [1]:
!apt-get update -y
!apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,597 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 kB]
Get:13 http://archive.ubuntu.com/ubuntu jam

In [2]:
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
!curl http://localhost:11434/api/tags

{"models":[]}

In [3]:
!ollama pull phi

In [4]:
!ollama list

NAME          ID              SIZE      MODIFIED       
phi:latest    e2fd6321a5fe    1.6 GB    32 seconds ago    


In [13]:
!apt-get install -y ffmpeg espeak-ng
!pip install faster-whisper kokoro>=0.9.4 soundfile gradio ffmpeg-python

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 110 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libsonic0 amd64 0.2.0-11build1 [10.3 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 espeak-ng-data amd64 1.50+dfsg-10ubuntu0.1 [3,956 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libespeak-ng1 amd64 1.50+dfsg-10ubuntu0.1 [207 kB]
Get:5 http://archive.ubuntu.c

In [18]:
print("⏳ Loading Whisper...")
# base = much better accuracy, still fast enough
stt = WhisperModel("base", device="cpu", compute_type="int8")

print("⏳ Loading Kokoro TTS...")
tts = KPipeline(lang_code='a')

print("⏳ Pre-warming TTS...")
_ = list(tts("ready", voice="af_sky", speed=1.0))

print("✅ All models loaded and warm!")

⏳ Loading Whisper...
⏳ Loading Kokoro TTS...


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


⏳ Pre-warming TTS...
✅ All models loaded and warm!


In [33]:
# Detect from USER's words — they show the real emotional context
USER_EMOTION_MAP = {
    "angry":     ["unacceptable", "frustrated", "furious", "terrible",
                  "worst", "ridiculous", "useless", "demand", "immediately"],
    "sad":       ["sad", "upset", "disappointed", "unhappy", "depressed",
                  "crying", "terrible", "horrible", "awful"],
    "happy":     ["happy", "love", "amazing", "wonderful", "great",
                  "fantastic", "excellent", "best", "thank you so much"],
    "excited":   ["excited", "thrilled", "can't wait", "awesome",
                  "incredible", "wow", "unbelievable"],
    "fearful":   ["worried", "scared", "afraid", "anxious", "concerned",
                  "serious", "urgent", "problem", "issue", "error", "broken"],
    "surprised": ["shocked", "unexpected", "didn't expect", "surprised",
                  "no way", "really", "wait"],
}

# User emotion → how AI should RESPOND emotionally
RESPONSE_EMOTION = {
    "angry":    "sad",       # AI responds apologetically
    "sad":      "sad",       # AI responds with empathy
    "happy":    "happy",     # AI responds cheerfully
    "excited":  "excited",   # AI matches energy
    "fearful":  "calm",      # AI responds calmly to reassure
    "surprised":"surprised", # AI matches surprise
    "neutral":  "neutral",
}

def detect_emotion(user_text, ai_text):
    # Step 1 — check user's words first (most reliable signal)
    u = user_text.lower()
    for emotion, keywords in USER_EMOTION_MAP.items():
        if any(k in u for k in keywords):
            ai_emotion = RESPONSE_EMOTION[emotion]
            print(f"   → User emotion: {emotion} → AI voice: {ai_emotion}")
            return ai_emotion

    # Step 2 — fallback to AI response text
    a = ai_text.lower()
    AI_KEYWORD_MAP = {
        "sad":   ["sorry","apologize","unfortunately","regret","disappoint"],
        "happy": ["glad","wonderful","pleased","delighted","great news"],
        "calm":  ["please note","kindly","let me","here is","certainly"],
    }
    for emotion, keywords in AI_KEYWORD_MAP.items():
        if any(k in a for k in keywords):
            print(f"   → AI text emotion: {emotion}")
            return emotion

    print(f"   → No keywords matched → neutral")
    return "neutral"

In [34]:
# ── STT ──────────────────────────────────────────────────────
def transcribe(filepath):
    converted = "_converted.wav"
    subprocess.run([
        "ffmpeg", "-y",
        "-i", filepath,
        "-ar", "16000",
        "-ac", "1",
        "-af", "afftdn=nf=-25",   # ✅ noise reduction before transcription
        "-f", "wav",
        converted
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    segments, _ = stt.transcribe(
        converted,
        beam_size=5,              # ✅ was 1 (fast but wrong), 5 = accurate
        language="en",
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=200),
        condition_on_previous_text=False,  # ✅ stops it carrying wrong words forward
        temperature=0.0            # ✅ deterministic = no random hallucination
    )
    return " ".join(s.text for s in segments).strip()

# ── LLM ──────────────────────────────────────────────────────
def ask_ai(user_text):
    res = requests.post("http://localhost:11434/api/generate", json={
        "model": "phi",
        "prompt": (
            "You are an empathetic AI call agent.\n"
            "Understand the user's emotion and reply accordingly.\n"
            "ONE complete sentence only. End with . ! or ?\n\n"
            f"User: {user_text}\n"
            "AI:"
        ),
        "stream": False,
        "options": {
            "num_predict": 80,
            "temperature": 0.8,
            "stop": [".", "!", "?"]
        }
    })
    return res.json()["response"].strip()

# ── TTS ──────────────────────────────────────────────────────
def speak(text, emotion):
    cfg = EMOTION_CONFIG.get(emotion, EMOTION_CONFIG["neutral"])
    chunks = [a for _, _, a in tts(
        cfg["prefix"] + text,
        voice=cfg["voice"],
        speed=cfg["speed"]
    )]
    return (24000, np.concatenate(chunks))

# ── MAIN PIPELINE ─────────────────────────────────────────────
def run_pipeline(file):
    if file is None:
        return "No file uploaded", "", "neutral", None, "No timing"

    filepath = file.name if hasattr(file, 'name') else file
    t0 = time.time()

    t = time.time()
    user_text = transcribe(filepath)
    stt_time = time.time() - t

    if not user_text:
        return "Could not understand audio", "", "neutral", None, "STT failed"

    t = time.time()
    ai_text = ask_ai(user_text)
    llm_time = time.time() - t

    emotion = detect_emotion(user_text, ai_text)   # ✅ pass both


    t = time.time()
    audio_out = speak(ai_text, emotion)
    tts_time = time.time() - t

    total = time.time() - t0

    # Timing summary for UI
    timing = (
        f"🎤 STT:      {stt_time*1000:.0f} ms\n"
        f"🤖 LLM:      {llm_time*1000:.0f} ms\n"
        f"🔊 TTS:      {tts_time*1000:.0f} ms\n"
        f"─────────────────\n"
        f"⚡ TOTAL:    {total*1000:.0f} ms  ({total:.2f}s)"
    )

    print(timing)
    return user_text, ai_text, emotion, audio_out, timing

In [35]:
with gr.Blocks(title="AI Call Agent") as demo:
    gr.Markdown("# 📞 AI Call Agent\nUpload audio → AI replies with emotion")

    with gr.Row():
        audio_in = gr.File(label="🎤 Upload your voice")
        btn = gr.Button("📞 Get AI Response", variant="primary", scale=0)

    with gr.Row():
        u = gr.Textbox(label="🧑 You said")
        a = gr.Textbox(label="🤖 AI Response")
        e = gr.Textbox(label="🎭 Emotion")

    # ✅ Timing panel
    timing_box = gr.Textbox(
        label="⏱️ Processing Time",
        lines=5,
        interactive=False,
        elem_id="timing"
    )

    spk = gr.Audio(label="🔊 AI Voice", autoplay=True)

    btn.click(
        fn=run_pipeline,
        inputs=[audio_in],
        outputs=[u, a, e, spk, timing_box]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://78e5312a41221c24d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
